# FraudLens: Notebook 01 — Exploratory Data Analysis & Data Profiling
### AI/ML & Data Science Pipeline for UPI Scam & Financial Threat Detection

This notebook conducts rigorous exploratory data analysis (EDA) on the **FraudLens Unified Cybercrime & SMS Dataset (5,228 records)**.
We analyze statistical distributions, class imbalances, stylometric characteristics, and linguistic signatures distinguishing genuine communications from coercive scam solicitations.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Load Unified Dataset
dataset_path = '../ai_engine/data/processed/combined_fraud_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Total Dataset Records: {len(df)}")
df.head(5)


## 1. Class Distribution Analysis
We evaluate the class distribution between genuine notifications ($y=0$) and fraudulent solicitations ($y=1$).
In production fraud detection, class imbalance is a foundational consideration.


In [ ]:
class_counts = df['label'].value_counts()
print(class_counts)
print(f"\nScam Percentage: {(class_counts['scam']/len(df))*100:.2f}%")

plt.figure(figsize=(7, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=['#10b981', '#f43f5e'])
plt.title('FraudLens Class Distribution: Genuine vs Scam', fontsize=12, fontweight='bold')
plt.ylabel('Number of Samples')
plt.show()


## 2. Stylometric & Message Length Analysis
Scam messages frequently exhibit distinct structural profiles (urgency padding, long advisory disclaimers, contact numbers).


In [ ]:
scam_df = df[df['target'] == 1]
safe_df = df[df['target'] == 0]

print(f"Scam Mean Character Length: {scam_df['char_length'].mean():.1f} | Safe Mean: {safe_df['char_length'].mean():.1f}")
print(f"Scam Mean Word Count: {scam_df['word_count'].mean():.1f} | Safe Mean: {safe_df['word_count'].mean():.1f}")

plt.figure(figsize=(9, 4))
sns.kdeplot(scam_df['char_length'], color='#f43f5e', fill=True, label='Scam (Threat)', alpha=0.4)
sns.kdeplot(safe_df['char_length'], color='#10b981', fill=True, label='Genuine (Safe)', alpha=0.4)
plt.title('Character Length Density Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Character Length')
plt.xlim(0, 300)
plt.legend()
plt.show()


## 3. Top Discriminative Tokens
We inspect high-frequency unigrams across scam payloads (excluding generic English stop words).


In [ ]:
words = []
stop = {'the', 'to', 'you', 'your', 'and', 'for', 'is', 'in', 'of', 'on', 'a', 'our', 'have', 'from', 'with', 'now'}
for t in scam_df['text']:
    tokens = re.findall(r'\b[a-zA-Z]{3,}\b', str(t).lower())
    words.extend([w for w in tokens if w not in stop])

top_20 = Counter(words).most_common(20)
kws, freqs = zip(*top_20)

plt.figure(figsize=(10, 5))
sns.barplot(x=list(freqs), y=list(kws), palette='flare')
plt.title('Top 20 Frequent Scam Keywords in Dataset', fontsize=12, fontweight='bold')
plt.xlabel('Frequency')
plt.show()
